In [0]:
# Databricks notebook source

# COMMAND ----------
# 1. Retrieve job parameters (Passed via Databricks Asset Bundles/Job UI)
source_path = dbutils.widgets.get("source_path")
catalog     = dbutils.widgets.get("catalog")
schema      = dbutils.widgets.get("schema")
table_name  = dbutils.widgets.get("table_name")

target_table = f"{catalog}.{schema}.{table_name}"

# COMMAND ----------
# 2. Ensure target catalog schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

# COMMAND ----------
# 3. Create Bronze Delta table with enterprise metadata and automatic audit columns
# Explicit column definitions ensure metadata comments are registered in Unity Catalog.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {target_table} (
    `Transaction unique identifier` STRING COMMENT 'Unique identifier for each transaction record',
    `Price` BIGINT COMMENT 'Transaction sale price in GBP',
    `Date of Transfer` STRING COMMENT 'Timestamp of property transfer',
    `Property Type` STRING COMMENT 'Property type: D=Detached, S=Semi, T=Terraced, F=Flats, O=Other',
    `Old/New` STRING COMMENT 'New build flag: Y=New, N=Established',
    `Duration` STRING COMMENT 'Tenure type: F=Freehold, L=Leasehold',
    `Town/City` STRING COMMENT 'Town or city location',
    `District` STRING COMMENT 'District administrative area',
    `County` STRING COMMENT 'County administrative area',
    `PPDCategory Type` STRING COMMENT 'Price Paid Data category type',
    `Record Status - monthly file only` STRING COMMENT 'Record status code',
    `load_dt` TIMESTAMP DEFAULT current_timestamp() COMMENT 'Audit: Ingestion timestamp into Bronze layer',
    `source` STRING DEFAULT '{source_path}' COMMENT 'Audit: Source file origin path'
)
USING DELTA
COMMENT 'Bronze transactions table loaded via COPY INTO from landing volume';
""")

# COMMAND ----------
# 4. Ingest CSV data directly using idempotent COPY INTO
# mergeSchema = 'true' ensures any net-new columns in future CSVs are dynamically appended to the Delta table.
spark.sql(f"""
COPY INTO {target_table}
FROM '{source_path}'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');
""")